# Day 5 - Table Transformer and PP-StructureV3 on Real Moderna Protocol Tables

This notebook runs the optional deep table models on the **same 20 actual public protocol table crops** and annotations used by the local Camelot/OpenCV benchmark. It does not substitute a synthetic dataset.

In [ ]:
# In Google Colab, upload and unzip regdoc-ai-day5.zip first, then set the path below.
from pathlib import Path
PROJECT_ROOT = Path('/content/regdoc-ai-current')
%cd {PROJECT_ROOT}
!pip -q install -e .
!pip -q install "transformers>=4.45,<5" accelerate timm
# PaddleOCR requires PaddlePaddle 3.x. Use CPU here; choose the official GPU wheel for your Colab CUDA runtime if desired.
!pip -q install paddlepaddle==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!pip -q install "paddleocr>=3,<4"

In [ ]:
import json
import time
import cv2
import pandas as pd
import torch
from PIL import Image
from transformers import AutoImageProcessor, TableTransformerForObjectDetection

from regdoc_ai.tables.metrics import boundary_metrics
from regdoc_ai.tables.geometry import merge_nearby

manifest = pd.read_csv('data/processed/table_benchmark/manifest.csv')
print(f"Tables: {len(manifest)}, NCT studies: {manifest.nct_id.nunique()}")
manifest.head()

## Table Transformer structure recognition

The images are already table crops, so this stage evaluates the structure-recognition model directly. Rows and columns are converted into boundary coordinates and compared with the PDF-derived reference annotations.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'microsoft/table-transformer-structure-recognition-v1.1-all'
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = TableTransformerForObjectDetection.from_pretrained(MODEL_ID).to(DEVICE).eval()
print(DEVICE, model.config.id2label)

In [ ]:
def boundary_values(boxes, labels, scores, id2label, axis, threshold=0.7):
    values=[]
    for box, label, score in zip(boxes.tolist(), labels.tolist(), scores.tolist()):
        name=id2label[int(label)]
        if score < threshold:
            continue
        x0,y0,x1,y1=box
        if axis=='x' and name=='table column':
            values.extend([round(x0), round(x1)])
        if axis=='y' and name=='table row':
            values.extend([round(y0), round(y1)])
    return merge_nearby(values, tolerance=8)

rows=[]
for item in manifest.to_dict(orient='records'):
    ann=json.load(open(item['annotation_path'], encoding='utf-8'))
    image=Image.open(item['image_path']).convert('RGB')
    inputs=processor(images=image, return_tensors='pt').to(DEVICE)
    started=time.perf_counter()
    with torch.no_grad():
        outputs=model(**inputs)
    target_sizes=torch.tensor([image.size[::-1]], device=DEVICE)
    result=processor.post_process_object_detection(outputs, threshold=0.55, target_sizes=target_sizes)[0]
    latency=time.perf_counter()-started
    pred_x=boundary_values(result['boxes'], result['labels'], result['scores'], model.config.id2label, 'x')
    pred_y=boundary_values(result['boxes'], result['labels'], result['scores'], model.config.id2label, 'y')
    gt_x=[round(v) for v in ann['x_boundaries_image']]
    gt_y=[round(v) for v in ann['y_boundaries_image']]
    xs=boundary_metrics(pred_x, gt_x, tolerance=10)
    ys=boundary_metrics(pred_y, gt_y, tolerance=10)
    rows.append({'engine':'table_transformer_v1.1_all','table_id':item['table_id'],
                 'row_boundary_f1':ys.f1,'column_boundary_f1':xs.f1,
                 'predicted_rows':max(0,len(pred_y)-1),'predicted_columns':max(0,len(pred_x)-1),
                 'reference_rows':ann['logical_rows'],'reference_columns':ann['logical_columns'],
                 'shape_exact':int(len(pred_y)-1==ann['logical_rows'] and len(pred_x)-1==ann['logical_columns']),
                 'latency_seconds':latency})

tatr=pd.DataFrame(rows)
out=Path('results/table_extraction_benchmark/table_transformer')
out.mkdir(parents=True, exist_ok=True)
tatr.to_csv(out/'table_predictions.csv', index=False)
tatr.agg({'shape_exact':'mean','row_boundary_f1':'mean','column_boundary_f1':'mean','latency_seconds':'mean'})

## PaddleOCR PP-StructureV3

The official pipeline saves JSON and Markdown outputs for every real table image. Its result schema can vary across PaddleOCR 3.x releases, so the raw outputs are preserved first; adapt the parser to the installed version before adding metrics to the final comparison.

In [ ]:
from paddleocr import PPStructureV3

pipeline = PPStructureV3(use_doc_orientation_classify=False, use_doc_unwarping=False)
out = Path('results/table_extraction_benchmark/ppstructurev3')
out.mkdir(parents=True, exist_ok=True)
status=[]
for item in manifest.to_dict(orient='records'):
    table_out=out/item['table_id']
    table_out.mkdir(parents=True, exist_ok=True)
    started=time.perf_counter()
    results=list(pipeline.predict(input=item['image_path']))
    latency=time.perf_counter()-started
    for result in results:
        result.save_to_json(save_path=str(table_out))
        result.save_to_markdown(save_path=str(table_out))
    status.append({'table_id':item['table_id'],'result_count':len(results),'latency_seconds':latency})
pd.DataFrame(status).to_csv(out/'run_status.csv', index=False)
pd.DataFrame(status).head()

## Completion guard

Only merge deep-model metrics into the project summary after all 20 tables complete. Do not replace unavailable or failed runs with zeros or invented values.

In [ ]:
assert len(tatr) == len(manifest) == 20
print('Table Transformer complete on all 20 tables.')
print('PP-StructureV3 raw outputs:', len(status), 'tables')